In [28]:
from typing import TypedDict, Annotated, Literal
import sqlite3
import requests

from dotenv import load_dotenv
from pydantic import BaseModel
from langgraph.graph import MessagesState

from langchain_core.runnables import RunnableConfig
from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, SystemMessage,HumanMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.base import BaseStore
from langchain_ollama import OllamaEmbeddings
load_dotenv()
import os


In [29]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)


In [30]:
from langgraph.store.memory import InMemoryStore
store=InMemoryStore()


In [31]:
user_id='u1'
user_details=("user", user_id,"details")


In [32]:
## adding memories
store.put(user_details,"1", {"data": "user likes waffle"})
store.put(user_details, "2", {"data": "user likes to play football"})
store.put(user_details, "3", {"data": "user likes pizza"})
store.put(user_details, "4", {"data": "user likes to watch series"})
store.put(user_details, "5", {"data": "user is based in india"})
store.put(user_details, "6", {"data": "user likes to watch cricket also"})
store.put(user_details, "7", {"data": "user favourite cricket player is virat kohli"})

In [33]:
system_prompt_template="""you are a helpful assistant that can answer questions about the user based on their memories.
You have access to the following memories about the user:
your goal is to answer the questions based on the memories you have access to. If you do not have enough information to answer the question, you should say "I don't know" and not make up an answer.
if the user name or relevant inforamtion is available in the memories then always personalize the answer with the user name or relevant information.
refrrance known works and projects and hobbies 
always insure that personal information is of the user and not of any other person.
int the end suggest one or two related question not always where it need there only
the users memory (which may be empty) is provided as {user_details_content}"""


In [34]:
def chat_node(state:MessagesState,config:RunnableConfig,store:BaseStore):
    user_id=config["configurable"]["user_id"]
    user_details=("user", user_id,"details")
    items=store.search(user_details)

    if items:
        user_details_content="\n".join(f"-{it.value.get('data','')}" for it in items)
    else:
        user_details_content=""
    system_prompt=system_prompt_template.format(user_details_content=user_details_content)
    system_msg=SystemMessage(content=system_prompt)
    response=llm.invoke([system_msg]+state['messages'])
    return {"messages":[response]}
    

In [36]:
builder=StateGraph(MessagesState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

graph=builder.compile(store=store)

In [40]:
config={"configurable":{"user_id":"u1"}}
result=graph.invoke(
    {"messages":[{ "role": "user", "content": "what is a best game to play and what are my hobbies?" }]},
    config
)

In [41]:
print(result["messages"][-1].content)

Based on the memories I have access to, it seems that you enjoy playing football. 

As for games, I'm not sure which one would be the "best" as it's subjective. However, since you like playing football, you might enjoy sports video games like FIFA or other football simulation games.

Your hobbies seem to include:

1. Football
2. Watching series (I don't know if this is a specific genre or type of show)
3. Eating pizza (you mentioned that you like pizza)

If I had to suggest some related questions, I'd ask:

* Do you have a favorite football team?
* Have you watched any popular Indian TV series?
